# Final Project: BERT for Palestinian POS Tagging

This notebook extends Curras with a 244-sentence Palestinian Digital Campus Corpus. It trains the extended AraBERT model, trains a controlled Curras-only baseline, evaluates both models, and displays the comparison.

## 0. Load the project from GitHub
This cell makes the notebook self-contained when it is opened directly from GitHub in Google Colab.

In [ ]:
from pathlib import Path

if not Path('train_and_evaluate.py').exists():
    !git clone --depth 1 https://github.com/AliShahroor/HBKU-Assignment.git /content/HBKU-Assignment
    %cd /content/HBKU-Assignment/BERT-Palestinian-POS-Tagging
else:
    print('Project files are already available in', Path.cwd())

## 1. Install dependencies

In [ ]:
!pip -q install 'transformers==4.57.3' 'datasets==3.6.0' 'accelerate==1.12.0' 'scikit-learn==1.8.0' gdown
import transformers, datasets, accelerate, sklearn
print('transformers', transformers.__version__)
print('datasets', datasets.__version__)
print('accelerate', accelerate.__version__)
print('scikit-learn', sklearn.__version__)

## 2. Download the professor-provided Curras split
The Drive folder is the same public folder referenced in the original class notebook.

In [ ]:
!mkdir -p data
!gdown --folder 'https://drive.google.com/drive/folders/1azGC6MLbkB3FTt5uxK1XFFdpY7X2DYsD?usp=sharing' --output data/Curras

## 3. Build the Palestinian Digital Campus corpus
The generator uses seed 42 and recreates the extension and combined Curras files deterministically.

In [ ]:
!python build_digital_campus_corpus.py

## 4. Fine-tune the extended AraBERT model
This run trains on Curras plus the new corpus and evaluates on the unchanged Curras test and the held-out Digital Campus test.

In [ ]:
!python train_and_evaluate.py --epochs 2 --batch-size 12

## 5. Train the controlled Curras-only baseline
Only the training data changes. The checkpoint, labels, seed, hyperparameters, and test sets remain fixed.

In [ ]:
!python train_and_evaluate.py --training-data curras-only --epochs 2 --batch-size 12

## 6. Evaluate both models on the Palestinian challenge set

In [ ]:
!python build_challenge_set.py
!python evaluate_challenge_set.py
!python evaluate_challenge_set.py --model-dir final_model_curras_only --output-stem challenge_curras_only

## 7. Display the final comparison
The values below are recomputed from the JSON files produced by the two training runs and challenge evaluations.

In [8]:
import json
from pathlib import Path

extended = json.loads(Path('results.json').read_text(encoding='utf-8'))
baseline = json.loads(Path('results_curras_only.json').read_text(encoding='utf-8'))
extended_challenge = json.loads(Path('challenge_results.json').read_text(encoding='utf-8'))
baseline_challenge = json.loads(Path('challenge_curras_only_results.json').read_text(encoding='utf-8'))

def scores(run, challenge):
    evaluation = run['evaluation']
    return (run['dataset']['train_tokens'],
            evaluation['curras_test']['f1_micro'],
            evaluation['digital_campus_test']['f1_micro'],
            challenge['accuracy_micro_f1'])

base = scores(baseline, baseline_challenge)
ext = scores(extended, extended_challenge)
print(f"{'Training data':<27} {'Train tokens':>12}  {'Curras F1':>9}  {'Campus F1':>9}  {'Challenge F1':>12}")
print(f"{'Curras only':<27} {base[0]:>12,}  {base[1]:>9.4f}  {base[2]:>9.4f}  {base[3]:>12.4f}")
print(f"{'Curras + extension':<27} {ext[0]:>12,}  {ext[1]:>9.4f}  {ext[2]:>9.4f}  {ext[3]:>12.4f}")
print('\nExtension minus baseline:')
print(f'Curras F1:    {ext[1] - base[1]:+.4f}')
print(f'Campus F1:    {ext[2] - base[2]:+.4f}')
print(f'Challenge F1: {ext[3] - base[3]:+.4f}')

Training data               Train tokens  Curras F1  Campus F1  Challenge F1
Curras only                       44,600     0.9408     0.9055        0.9101
Curras + extension                46,588     0.9424     0.9370        0.8989

Extension minus baseline:
Curras F1:    +0.0016
Campus F1:    +0.0315
Challenge F1: -0.0112
